In [1]:
import numpy as np
import pandas as pd

from blox2 import BLOX2Selector, split_df_by_n_rows
from blox2.predictor import RidgePredictor, LassoPredictor, RandomForestPredictor, SVRPredictor

# predictor = RidgePredictor()
# predictor = LassoPredictor()
predictor = RandomForestPredictor(n_estimators=100)
# predictor = SVRPredictor()

observed_features, unchecked_features = split_df_by_n_rows(pd.read_csv("../data/zinc_dft/feature_list.csv", header=None), 10)
observed_values, unchecked_values = split_df_by_n_rows(pd.read_csv("../data/zinc_dft/properties.csv"), 10)

# observed_features = pd.read_csv("../data/test/feature_list_of_observed_data.csv", header=None)
# unchecked_features = pd.read_csv("../data/test/feature_list_of_unchecked_data.csv", header=None)
# observed_values = pd.read_csv("../data/test/properties_of_observed_data.csv")
# unchecked_values = pd.read_csv("../data/test/properties_of_unchecked_data.csv")

In [ ]:
import time

n_iters = 2000
report_interval = 10

selector = BLOX2Selector(observed_features, observed_values, unchecked_features, predictor, squared_sigma=1, normalize_features=False, n_obs_samples=None, normalize_values=True, n_chunks=512, use_distribution=False, verbose=True, compare_selection_time=False)
t0 = time.perf_counter()

for i in range(n_iters):
    ids = selector.next_candidates(n=1)
    for id in ids:
        values = np.asarray(unchecked_values.iloc[id - len(observed_features)])
        selector.observe(id, values)
    if (i+1) % report_interval == 0:
        print(f"{i+1} candidates suggested. Passed time: {time.perf_counter() - t0}")

10 candidates suggested. Passed time: 10.544676700024866
20 candidates suggested. Passed time: 21.245376700011548
30 candidates suggested. Passed time: 32.17389520001598
40 candidates suggested. Passed time: 43.40250269998796
50 candidates suggested. Passed time: 54.508424300001934
60 candidates suggested. Passed time: 65.52674110000953
70 candidates suggested. Passed time: 76.63255510001909
80 candidates suggested. Passed time: 87.83866599999601
90 candidates suggested. Passed time: 99.18318970000837
100 candidates suggested. Passed time: 110.830253800028
110 candidates suggested. Passed time: 122.39218889997574
120 candidates suggested. Passed time: 134.18863769999007
130 candidates suggested. Passed time: 146.0523695999873
140 candidates suggested. Passed time: 158.17156500002602
150 candidates suggested. Passed time: 170.37883230001898
160 candidates suggested. Passed time: 182.76573939999798
170 candidates suggested. Passed time: 195.33008560002781
180 candidates suggested. Passed

In [ ]:
import matplotlib.pyplot as plt

def plot_lists(*lists, labels=None):
    if len(lists) == 0:
        raise ValueError("At least one list must be provided.")

    length = len(lists[0])
    x = range(length)

    for i, lst in enumerate(lists):
        label = labels[i] if labels is not None else None
        plt.plot(x, lst, label=label)

    plt.xlabel("iter")
    plt.ylabel("time (sec)")
    if labels is not None:
        plt.legend()
    plt.tight_layout()
    plt.show()
    
def add_lists(a: list[float], b: list[float]) -> list[float]:
    return [x + y for x, y in zip(a, b)]

if selector.compare_selection_time:
    plot_lists(selector.passed_times_blox2[:12], selector.passed_times_repli[:12], add_lists(selector.passed_times_train[:12], selector.passed_times_pred[:12]), labels=["selection_new", "selection_replication", "train + pred"])
else:
    plot_lists(selector.passed_times_selection, add_lists(selector.passed_times_train, selector.passed_times_pred), labels=["selection", "train + pred"])

In [ ]:
sum(selector.passed_times_blox2[:12])

In [ ]:
sum(selector.passed_times_repli[:12])